# E3 -- extended Muller-Brown (10D): plot notebook

**This notebook only reads saved results.**

It must not call a sampler, a PT tuner, an LSC quadrature refinement, a `dt` refinement, or a reference builder, and it must not recompute any official metric. Every number drawn here already exists in a run's `metrics_timeseries.csv` or `cost_timeseries.csv`, written by `E3_muller_brown_run.ipynb` at run time.

Scatter, CDF, histogram, and KDE panels are **display only**. They visualise the saved sample snapshots and **never override, correct, or stand in for** the numbers in `metrics_timeseries.csv`. If a picture and a saved metric disagree, the saved metric is the result.

Method colours, markers, and display names come from `configs/registry.yaml`; which runs to draw and how to lay them out comes from `configs/plots/manuscript.yaml`. Neither table is redefined here.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.catalog import select_runs
from src.plotting import (curve_figure, load_plot_config, load_runs,
                          save_figure, snapshot_figure)

REPO_ROOT = Path("..")
EXPERIMENT_ID = "E3"

plot_config = load_plot_config(REPO_ROOT / "configs" / "plots" / "manuscript.yaml")
defaults = plot_config["defaults"]
spec = plot_config[EXPERIMENT_ID]
figures = spec["figures"]

EXPERIMENT_DIR = REPO_ROOT / "results" / spec["experiment_key"]
OUTPUT_DIR = REPO_ROOT / defaults["output_root"] / spec["experiment_key"]
FORMATS = tuple(defaults["output_formats"])  # png, pdf, svg, tiff
FIGURES = {}


def load_available(experiment_dir, spec, **kwargs):
    """Load the spec's runs, tolerating methods this campaign could not run.

    Canonical (untamed) variants are expected to be unusable on several of
    these targets: the drift is not truncated, so at the step size the run
    stage settled on they are genuinely unstable. That is a result, not a plotting problem, and a
    figure must still draw the methods that did run. Uncalibratable methods are
    annotated by the plotting module; anything still missing here is dropped
    with a printed warning rather than aborting the notebook.
    """
    try:
        return load_runs(experiment_dir, spec, **kwargs)
    except ValueError as error:
        if "requires missing methods" not in str(error):
            raise
        print(f"warning: {error}")
        print("plotting only the runs that completed")
        # methods=None disables the all-methods-required check; the spec's
        # variant filters still exclude unrelated methods.
        return load_runs(experiment_dir, spec, methods=None, **kwargs)


print(f"{EXPERIMENT_ID}: {len(figures)} specified figures -> {OUTPUT_DIR}")

## What is plottable

Load the derived catalog (rebuilding it from the manifests if it is missing) and list the runs it admits, so it is visible up front which methods, variants, and step sizes this notebook can actually draw. Nothing is run here; this is a directory listing.

In [ ]:
runs = select_runs(EXPERIMENT_DIR, latest_only=defaults["latest_run_only"])

print(f"{len(runs)} plottable runs\n")
print(f"{'method':<12}{'variant label':<34}{'tame':<7}{'dt':<10}run id")
for row in runs:
    print(f"{row['method']:<12}{row['variant_label']:<34}"
          f"{str(row['tame']):<7}{str(row['dt']):<10}{row['run_id']}")

### Figure E3.1 -- latent CV samples over simulation time

**Rows** are FLA, PT, and LSC-CP. **Columns** are four matched simulation times. Each panel draws the reference collective-variable free-energy surface as contours, on the same levels in every panel, with that method's latent CV samples on top.

The CV is the **latent pair** $z$, never the first two sampling coordinates. No minima, saddle, basin-boundary, jump-arrow, or transition-path annotations: the figure shows a CV distribution, not a kinetic story.

In [ ]:
figure_spec = figures["E3.1_cv_snapshots_matched_time"]

FIGURES["E3.1_cv_snapshots_matched_time"] = snapshot_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E3.1_cv_snapshots_matched_time"]

### Figure E3.2 -- collective-variable agreement

A 2x2 grid. **Rows** are CV-SW$_2$ and CV-MMD$^2$; **columns** are against simulation time and against FEE.

In [ ]:
figure_spec = figures["E3.2_cv_metrics"]

FIGURES["E3.2_cv_metrics"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E3.2_cv_metrics"]

### Figure E3.3 -- supplement

Supplementary CV diagnostics: the fixed-bandwidth KDE squared Hellinger distance, the unbiased CV-MMD$^2$, and the latent $z_1$ marginal KS distance. The KDE panel is a display diagnostic and does not replace CV-SW$_2$ or CV-MMD$^2$.

In [ ]:
figure_spec = figures["E3.3_supplement"]

FIGURES["E3.3_supplement"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E3.3_supplement"]

### Figure E3.4 -- LSC score potential-evaluation cost

The LSC-only cost figure: full deterministic-quadrature LSC-CP against LSC-CP-RA(A). The x axis counts LSC score potential evaluations only and is not a complete computational cost.

In [ ]:
figure_spec = figures["E3.4_lsc_score_cost"]

FIGURES["E3.4_lsc_score_cost"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E3.4_lsc_score_cost"]

## Canonical, tamed, and paired views

The main curve figure is regenerated three times, from the same saved runs: **canonical only**, **tamed only**, and the **paired** canonical-versus-tamed overlay.

The convention, taken from `configs/registry.yaml` and the plot defaults:

* the **method** sets the **colour**, and taming never changes it;
* **canonical** is a **solid** line;
* **tamed** is a **dashed** line;
* **hyperparameter values** are distinguished by **marker**, and the value is written into the legend label.

So colour answers "which method", line style answers "tamed or not", and marker answers "which hyperparameter value".

In [ ]:
MAIN_CURVE_FIGURE = "E3.2_cv_metrics"

for view in defaults["tame_views"]:
    view_spec = {**figures[MAIN_CURVE_FIGURE], "tame_view": view}
    FIGURES[f"{MAIN_CURVE_FIGURE}__{view}"] = curve_figure(
        load_available(EXPERIMENT_DIR, view_spec), view_spec)

print("tame views:", list(defaults["tame_views"]))

## Export

Every figure built above is written to **PNG, PDF, SVG, and TIFF** under `figures/<experiment key>/`, from the format list in the plot defaults. Re-running this cell overwrites the files in place; it never touches anything under `results/`.

In [ ]:
for name, figure in FIGURES.items():
    save_figure(figure, name, OUTPUT_DIR, formats=FORMATS)
    print(f"saved {name}  [{', '.join(FORMATS)}]")

print(f"\n{len(FIGURES)} figures written under {OUTPUT_DIR}")